# Great Expectations — Step-by-Step Demo

This notebook walks through the **Great Expectations (GE)** workflow for data quality testing using the local `data.xlsx` (a Sales dataset).

**What you'll learn:**
1. How to install Great Expectations and its dependencies
2. How to load a pandas DataFrame as a GE Dataset
3. How to write data quality expectations (row counts, uniqueness, nulls, types, value sets, ranges, means)
4. How to save expectations to a JSON suite and re-run them on new data
5. How to react to validation results in a pipeline

> Reference API used: `great_expectations` **0.15.x** legacy `PandasDataset` API (matches the supplied reference notebook).

## Step 1 — Installation

Run the cell below **once** to install everything you need. The `%pip` magic installs into the kernel that is currently running this notebook.

**Packages:**
- `great_expectations==0.15.41` — the data-quality framework (pinned to match the reference notebook)
- `pandas` — DataFrame engine
- `openpyxl` — required by pandas to read `.xlsx` files
- `jupyter` — to run this notebook

**Equivalent shell commands (alternative):**
```bash
python3 -m venv .venv && source .venv/bin/activate     # optional: isolated env
pip install --upgrade pip
pip install great_expectations==0.15.41 pandas openpyxl jupyter
jupyter notebook                                       # then open this file
```

In [ ]:
%pip install --quiet great_expectations==0.15.41 pandas openpyxl

## Step 2 — Import libraries and load the data

We import `great_expectations` (aliased `ge`) and `pandas`, then read the local Excel file into a DataFrame.

In [ ]:
import great_expectations as ge
import pandas as pd

print('Great Expectations version:', ge.__version__)

In [ ]:
df = pd.read_excel('data.xlsx')
print('Shape:', df.shape)
df.head()

In [ ]:
# Quick look at column names and dtypes — useful when designing expectations
df.dtypes

## Step 3 — Convert the pandas DataFrame to a Great Expectations Dataset

`ge.from_pandas(df)` returns a `PandasDataset` — a subclass of `pd.DataFrame` that adds **`expect_*`** methods you can call directly on the data.

In [ ]:
ge_df = ge.from_pandas(df)
type(ge_df)

## Step 4 — Table-level expectations

Start with checks on the **whole table**, e.g. row counts. Each `expect_*` call returns a result object with:
- `success` — `True`/`False`
- `result` — the observed value(s)
- `exception_info` — populated if the check itself errored

In [ ]:
# How many rows do we expect? Start with what we observe, then constrain it.
observed_rows = len(ge_df)
print('Observed row count:', observed_rows)

ge_df.expect_table_row_count_to_be_between(min_value=1, max_value=1_000_000)

## Step 5 — Column-level expectations: existence, uniqueness, nulls, types

These are the bread-and-butter checks for a **primary/business key column**. We use the `product` column from the Sales dataset.

In [ ]:
# 5a. The column must exist
ge_df.expect_column_to_exist('product')

In [ ]:
# 5b. No null product names
ge_df.expect_column_values_to_not_be_null('product')

In [ ]:
# 5c. OrderQuantity should be an integer type
ge_df.expect_column_values_to_be_in_type_list('OrderQuantity', ['int', 'int64', 'int32'])

## Step 6 — Categorical checks: value-in-set

Use `expect_column_values_to_be_in_set` to enforce **allowed categories**. First peek at the unique values, then constrain them.

In [ ]:
df['productcategory'].unique()

In [ ]:
allowed_categories = ['Bikes', 'Components', 'Clothing', 'Accessories']
ge_df.expect_column_values_to_be_in_set('productcategory', allowed_categories)

## Step 7 — Numeric range checks: min, max, mean, per-row

Catch outliers and corrupted data with min/max/mean expectations.

In [ ]:
# Every order quantity must be at least 1 (no zero / negative orders)
ge_df.expect_column_values_to_be_between('OrderQuantity', min_value=1, max_value=1000)

In [ ]:
# `Discount` in this dataset is a monetary amount (not a 0-1 fraction),
# so the rule is: must be non-negative and within a sensible cap.
ge_df.expect_column_values_to_be_between('Discount', min_value=0, max_value=10_000)

In [ ]:
# Average UnitPrice should fall in a sensible business range
ge_df.expect_column_mean_to_be_between('UnitPrice', min_value=1, max_value=10_000)

In [ ]:
# Geographic sanity checks
ge_df.expect_column_values_to_be_between('latitude',  min_value=-90,  max_value=90)
ge_df.expect_column_values_to_be_between('longitude', min_value=-180, max_value=180)

## Step 8 — Save the expectation suite to JSON

All the expectations we ran above were collected into an **Expectation Suite** attached to `ge_df`. Persist it so it can be re-run on future data loads (CI checks, daily ETL, etc.).

In [ ]:
# Inspect the suite that was built up implicitly
suite = ge_df.get_expectation_suite(discard_failed_expectations=False)
print('Number of expectations:', len(suite.expectations))
suite

In [ ]:
ge_df.save_expectation_suite('sales.data.expectations.json', discard_failed_expectations=False)
print('Saved -> sales.data.expectations.json')

## Step 9 — Re-validate against the saved suite

Simulate a **next-day data refresh**: re-load the file and validate it against the saved JSON suite. In production the new data would come from your ETL job, an API, a database, etc.

In [ ]:
df_new = ge.read_excel('data.xlsx')                         # GE wrapper around pd.read_excel
results = df_new.validate(expectation_suite='sales.data.expectations.json')
results['statistics']

### Step 9b — Diagnose any failures

Before gating the pipeline, print a readable summary of every expectation that failed: which column, what was expected, and what was actually observed. This is what you'd typically log or post to Slack/email.

In [ ]:
stats = results['statistics']
print(f"Passed {stats['successful_expectations']}/{stats['evaluated_expectations']} expectations "
      f"({stats['success_percent']:.1f}%)\n")

for r in results['results']:
    if r['success']:
        continue
    cfg     = r['expectation_config']
    etype   = cfg['expectation_type']
    kwargs  = {k: v for k, v in cfg['kwargs'].items() if k != 'batch_id'}
    res     = r.get('result', {})
    print(f"FAIL: {etype}")
    print(f"  kwargs   : {kwargs}")
    if 'observed_value' in res:
        print(f"  observed : {res['observed_value']}")
    if 'unexpected_count' in res:
        print(f"  unexpected_count   : {res['unexpected_count']} "
              f"({res.get('unexpected_percent', 0):.2f}%)")
    if res.get('partial_unexpected_list'):
        print(f"  sample bad values : {res['partial_unexpected_list'][:5]}")
    print()

## Step 10 — Use the result to gate your pipeline

A typical ETL/ML pipeline either **continues** when all checks pass or **fails fast** when any check breaks — preventing bad data from reaching downstream systems.

The `Exception` raised below now includes the **column name** so you can immediately see what to fix or relax.

In [ ]:
if results['success']:
    print('All Data Quality Tests passed — safe to proceed downstream.')
else:
    failed = [
        {
            'expectation': r['expectation_config']['expectation_type'],
            'column'     : r['expectation_config']['kwargs'].get('column'),
            'observed'   : r.get('result', {}).get('observed_value'),
            'bad_count'  : r.get('result', {}).get('unexpected_count'),
        }
        for r in results['results'] if not r['success']
    ]
    raise Exception(f'Data quality failures: {failed}')

## Step 11 — Data Docs (the built-in GUI)

**Data Docs** are HTML reports auto-generated by Great Expectations. They show every suite, every validation run, observed values, and sample failing rows — all clickable.

We need a small `DataContext` (the configuration object that knows where to store suites/results/docs). To keep Colab simple we use an **ephemeral** in-memory context that writes only the docs to disk.

In [ ]:
import os, json
from great_expectations.data_context import BaseDataContext
from great_expectations.data_context.types.base import (
    DataContextConfig, FilesystemStoreBackendDefaults,
)
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.core.expectation_configuration import ExpectationConfiguration
from great_expectations.core.batch import RuntimeBatchRequest

# 1. Spin up a tiny on-disk context (works equally on Colab or laptop)
GX_DIR = os.path.abspath('gx')
ctx = BaseDataContext(project_config=DataContextConfig(
    store_backend_defaults=FilesystemStoreBackendDefaults(root_directory=GX_DIR),
))

# 2. Register a runtime pandas datasource so the context can validate our DataFrame
ctx.add_datasource(
    name='pandas_runtime',
    class_name='Datasource',
    execution_engine={'class_name': 'PandasExecutionEngine'},
    data_connectors={
        'runtime_connector': {
            'class_name': 'RuntimeDataConnector',
            'batch_identifiers': ['run_id'],
        }
    },
)

# 3. Re-register our saved suite inside the context
with open('sales.data.expectations.json') as f:
    suite_dict = json.load(f)
suite = ExpectationSuite(expectation_suite_name='sales_suite', data_context=ctx)
for e in suite_dict['expectations']:
    suite.add_expectation(ExpectationConfiguration(**e))
ctx.save_expectation_suite(suite)

# 4. Build a Checkpoint that pairs the suite with a batch of our DataFrame.
#    Checkpoints are what persist validation results into the docs.
batch_request = RuntimeBatchRequest(
    datasource_name='pandas_runtime',
    data_connector_name='runtime_connector',
    data_asset_name='sales_data',
    runtime_parameters={'batch_data': df},
    batch_identifiers={'run_id': 'demo_run'},
)
# GE 0.15.41 only ships add_checkpoint (no add_or_update_*), so make the cell
# re-run safe by deleting any prior checkpoint of the same name first.
# Note: a RuntimeBatchRequest carries the actual DataFrame in `batch_data`,
# which cannot be JSON-serialised into the checkpoint store. So we register
# the checkpoint WITHOUT validations and pass the batch_request at run time.
checkpoint_name = 'sales_checkpoint'
if checkpoint_name in ctx.list_checkpoints():
    ctx.delete_checkpoint(checkpoint_name)
ctx.add_checkpoint(
    name=checkpoint_name,
    config_version=1,
    class_name='SimpleCheckpoint',
    expectation_suite_name='sales_suite',
)
ckpt_result = ctx.run_checkpoint(
    checkpoint_name=checkpoint_name,
    validations=[{'batch_request': batch_request,
                  'expectation_suite_name': 'sales_suite'}],
)
ctx.build_data_docs()

docs_index = os.path.join(GX_DIR, 'uncommitted/data_docs/local_site/index.html')
print('Checkpoint success :', ckpt_result['success'])
print('Data Docs built at :', docs_index)

In [ ]:
# The most reliable way to view Data Docs is to serve the folder over HTTP
# and embed the result. This works in BOTH Colab and local Jupyter — relative
# CSS/JS resolve correctly because the browser loads it as a real web page.
import http.server, socketserver, threading, glob
from IPython.display import IFrame, display

DOCS_ROOT = os.path.join(GX_DIR, 'uncommitted/data_docs/local_site')
PORT = 8765

# Start a background HTTP server rooted at the docs folder (idempotent — re-run safe)
if not getattr(globals().get('_gx_httpd', None), 'serving', False):
    class _Handler(http.server.SimpleHTTPRequestHandler):
        def __init__(self, *a, **kw): super().__init__(*a, directory=DOCS_ROOT, **kw)
        def log_message(self, *a, **kw): pass
    _gx_httpd = socketserver.TCPServer(('', PORT), _Handler)
    _gx_httpd.serving = True
    threading.Thread(target=_gx_httpd.serve_forever, daemon=True).start()
    print(f'Serving Data Docs at http://localhost:{PORT}/')

# Pick the most recent validation result page (richest view: green/red grid + samples)
validation_pages = sorted(
    glob.glob(os.path.join(DOCS_ROOT, 'validations/**/*.html'), recursive=True),
    key=os.path.getmtime,
)
rel = os.path.relpath(validation_pages[-1], DOCS_ROOT) if validation_pages else 'index.html'
print('Showing:', rel)

try:
    # Colab: route through Colab's built-in port proxy (works around iframe sandboxing)
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, path='/' + rel, height='900')
except ImportError:
    # Local Jupyter / VS Code
    display(IFrame(src=f'http://localhost:{PORT}/{rel}', width='100%', height=900))

In [ ]:
# Optional: zip the whole report and download it (works in Colab + Jupyter)
import shutil
shutil.make_archive('data_docs', 'zip', os.path.join(GX_DIR, 'uncommitted/data_docs/local_site'))
print('Wrote data_docs.zip — unzip and open index.html in any browser.')

# In Colab specifically:
try:
    from google.colab import files
    files.download('data_docs.zip')
except ImportError:
    pass

## Step 12 — Stress test: inject realistic failures

Until now every expectation passed because `data.xlsx` happens to be clean. To prove the suite actually catches problems, we make a **dirty copy** of the DataFrame and inject several failures that are *plausible* in real-world ETL pipelines:

| # | Injected fault                         | Realistic cause                                  | Rule it should trip                                              |
|---|----------------------------------------|--------------------------------------------------|------------------------------------------------------------------|
| 1 | 3 `product` rows set to `NaN`          | Upstream join lost the product lookup            | `expect_column_values_to_not_be_null('product')`                |
| 2 | 2 `productcategory` rows set to `'Toys'` | New category added in source, not yet whitelisted | `expect_column_values_to_be_in_set('productcategory', …)`       |
| 3 | 1 `OrderQuantity` set to `0`, 1 set to `5000` | Refund encoded as 0; bulk-order data-entry typo | `expect_column_values_to_be_between('OrderQuantity', 1, 1000)`  |
| 4 | 1 `Discount` set to `-25`              | Sign flipped during currency conversion          | `expect_column_values_to_be_between('Discount', 0, 10000)`      |
| 5 | 1 `latitude` set to `99.9`             | Lat/long swapped at source                       | `expect_column_values_to_be_between('latitude', -90, 90)`       |

Five separate column-level rules, five separate failure classes — exactly what GE is designed for.

In [ ]:
import numpy as np

df_dirty = df.copy()

# 1. Nulls in `product` (3 rows)
df_dirty.loc[df_dirty.index[:3], 'product'] = np.nan

# 2. Unknown category (2 rows)
df_dirty.loc[df_dirty.index[3:5], 'productcategory'] = 'Toys'

# 3. Out-of-range OrderQuantity
df_dirty.loc[df_dirty.index[5], 'OrderQuantity'] = 0      # refund-coded-as-zero
df_dirty.loc[df_dirty.index[6], 'OrderQuantity'] = 5000   # data-entry typo

# 4. Negative Discount
df_dirty.loc[df_dirty.index[7], 'Discount'] = -25

# 5. Impossible latitude (lat/long swapped)
df_dirty.loc[df_dirty.index[8], 'latitude'] = 99.9

print('Dirty rows preview:')
df_dirty.loc[df_dirty.index[:9], ['product', 'productcategory', 'OrderQuantity', 'Discount', 'latitude']]

In [ ]:
# Re-run the saved suite against the dirty data — same pattern as Step 9.
ge_dirty = ge.from_pandas(df_dirty)
dirty_results = ge_dirty.validate(expectation_suite='sales.data.expectations.json')

stats = dirty_results['statistics']
print(f"Passed {stats['successful_expectations']}/{stats['evaluated_expectations']} expectations "
      f"({stats['success_percent']:.1f}%)\n")

for r in dirty_results['results']:
    if r['success']:
        continue
    cfg     = r['expectation_config']
    etype   = cfg['expectation_type']
    kwargs  = {k: v for k, v in cfg['kwargs'].items() if k != 'batch_id'}
    res     = r.get('result', {})
    print(f"FAIL: {etype}")
    print(f"  kwargs   : {kwargs}")
    if 'observed_value' in res:
        print(f"  observed : {res['observed_value']}")
    if 'unexpected_count' in res:
        print(f"  unexpected_count   : {res['unexpected_count']} "
              f"({res.get('unexpected_percent', 0):.2f}%)")
    if res.get('partial_unexpected_list'):
        print(f"  sample bad values : {res['partial_unexpected_list'][:5]}")
    print()

### How GE caught each issue

Every injected fault surfaces as an **independent** failed expectation in the report above — GE keeps evaluating the rest of the suite even after one rule fails, so a single run gives you the **complete diagnostic picture** instead of stopping at the first error.

- **`unexpected_count`** tells you *how many rows* violate the rule (3 nulls, 2 bad categories, 2 bad quantities, 1 bad discount, 1 bad latitude).
- **`partial_unexpected_list`** gives you up to 20 sample bad values — perfect for a Slack alert or a Jira ticket.
- **`observed_value`** on aggregate rules (e.g. mean) shows the actual computed metric vs. the expected range.

Run the next cell to push the dirty validation through the same Checkpoint and view the failures in the HTML report.

In [ ]:
# Push the dirty DataFrame through the same Checkpoint so the failures land
# in Data Docs, then re-display the latest validation page.
# Requires that Step 11 has already been run (defines ctx, checkpoint_name,
# DOCS_ROOT, PORT and starts the background HTTP server).
import glob
from great_expectations.core.batch import RuntimeBatchRequest
from IPython.display import IFrame, display

dirty_batch_request = RuntimeBatchRequest(
    datasource_name='pandas_runtime',
    data_connector_name='runtime_connector',
    data_asset_name='sales_data_dirty',
    runtime_parameters={'batch_data': df_dirty},
    batch_identifiers={'run_id': 'dirty_run'},
)
dirty_ckpt_result = ctx.run_checkpoint(
    checkpoint_name=checkpoint_name,
    validations=[{'batch_request': dirty_batch_request,
                  'expectation_suite_name': 'sales_suite'}],
)
ctx.build_data_docs()
print('Dirty checkpoint success :', dirty_ckpt_result['success'])

# Pick the most-recent validation page (the dirty run we just produced)
validation_pages = sorted(
    glob.glob(os.path.join(DOCS_ROOT, 'validations/**/*.html'), recursive=True),
    key=os.path.getmtime,
)
rel = os.path.relpath(validation_pages[-1], DOCS_ROOT)
print('Showing :', rel)

try:
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(PORT, path='/' + rel, height='900')
except ImportError:
    display(IFrame(src=f'http://localhost:{PORT}/{rel}', width='100%', height=900))

---
# Part 2 — Advanced Features

Steps 13–17 showcase capabilities that distinguish GE from a hand-rolled `assert` script: auto-generated suites, statistical drift detection, cross-column integrity, format/regex rules, and custom domain expectations. All run on the **same `data.xlsx`** loaded as `df` in Step 2 — no extra files needed.

## Step 13 — Auto-profile a starter suite

Hand-writing 10 expectations took us most of Steps 4–7. The **`UserConfigurableProfiler`** reads a *trusted* snapshot of the data and writes an entire suite for you in one call — types, ranges, null %, value sets, mean, quantiles. Use it to onboard a new table in seconds, then prune/edit the result.

We sample 5 000 rows so it runs in ~2 seconds; on a real warehouse you'd profile a representative day's data.

In [ ]:
from great_expectations.profile.user_configurable_profiler import UserConfigurableProfiler
from collections import Counter

# Drop the unnamed trailing column (its header is the integer 11, which breaks
# the profiler's column-sort step that assumes string column names)
df_for_profile = df.drop(columns=[11], errors='ignore')
sample_ds = ge.from_pandas(df_for_profile.sample(n=5000, random_state=42))

profiler = UserConfigurableProfiler(
    profile_dataset=sample_ds,
    not_null_only=False,             # also propose ranges, sets, etc.
    value_set_threshold='MANY',      # only propose value sets for low-cardinality cols
)
auto_suite = profiler.build_suite()

print(f'\nProfiler generated {len(auto_suite.expectations)} expectations from one snapshot\n')
print('Top expectation types:')
for etype, n in Counter(e.expectation_type for e in auto_suite.expectations).most_common(8):
    print(f'  {n:3d}  {etype}')

In [ ]:
# Persist the auto-suite as a starting point you can hand-edit later
import json
with open('sales.auto.expectations.json', 'w') as f:
    json.dump(auto_suite.to_json_dict(), f, indent=2)
print('Saved -> sales.auto.expectations.json   ({} bytes)'.format(
    __import__('os').path.getsize('sales.auto.expectations.json')))

## Step 14 — Drift detection with KL divergence

Min/max checks miss the most insidious failures: data that is *technically* in range but whose **distribution has shifted**. Classic example — a model trained on last quarter's orders silently degrades because the price mix changed.

GE's `expect_column_kl_divergence_to_be_less_than` compares the live distribution against a saved baseline (a *partition object*) and fails when the divergence exceeds your threshold.

We:
1. Use the **full historical data** as the reference distribution of `UnitPrice` (covers all bin ranges).
2. Take 2013+ orders as the **current** batch.
3. Inject realistic drift by shifting `UnitPrice` up by **10%** (think: across-the-board price hike).
4. Watch GE flag it.

*Note*: `internal_weight_holdout=0.01` reserves a tiny weight for empty bins so the divergence stays finite when the live data has values not seen in the baseline.

In [ ]:
from great_expectations.dataset.util import build_continuous_partition_object

current_df = df[df['OrderDate'] >= '2013-01-01'].copy()
print(f'Reference rows: {len(df):,}    Current rows: {len(current_df):,}')

# 1. Compute the reference distribution of UnitPrice (auto-binned histogram)
reference_ds = ge.from_pandas(df)
partition    = build_continuous_partition_object(
    dataset=reference_ds, column='UnitPrice', bins='auto',
)
print(f'Reference partition: {len(partition["bins"])-1} bins')

# 2. Validate the un-shifted current data — KL should be small (in-distribution)
ok = ge.from_pandas(current_df).expect_column_kl_divergence_to_be_less_than(
    column='UnitPrice', partition_object=partition, threshold=0.5,
    internal_weight_holdout=0.01,
)
print(f"\nUn-shifted current data  -> success={ok['success']}, KL={ok['result']['observed_value']:.4f}")

# 3. Now inject a 10% pricing drift
drifted = current_df.copy()
drifted['UnitPrice'] = drifted['UnitPrice'] * 1.1
fail = ge.from_pandas(drifted).expect_column_kl_divergence_to_be_less_than(
    column='UnitPrice', partition_object=partition, threshold=0.5,
    internal_weight_holdout=0.01,
)
print(f"10% drifted current data -> success={fail['success']}, KL={fail['result']['observed_value']:.4f}")
print('\nA 10% across-the-board price shift -> KL jumps ~10x and trips the threshold.')

## Step 15 — Cross-column / multi-row integrity

Real data quality usually involves *relationships between columns*, not just per-column checks. GE supports this directly:

- **`expect_column_pair_values_A_to_be_greater_than_B`** — accounting/margin rules (e.g. `UnitPrice >= StandardCost`, no selling at a loss).
- **`expect_compound_columns_to_be_unique`** — composite primary keys (e.g. `(Customer, OrderDate, product, OrderQuantity)` should identify a unique line item).
- **`expect_column_values_to_be_increasing`** — ordering invariants (after sorting by date, dates must be monotonic — catches duplicates and time-travel bugs).

In [ ]:
# 1. Margin sanity: UnitPrice should never fall below StandardCost (ignoring promotional zero-price rows)
non_promo = df[df['UnitPrice'] > 0]
margin = ge.from_pandas(non_promo).expect_column_pair_values_A_to_be_greater_than_B(
    column_A='UnitPrice', column_B='StandardCost', or_equal=True,
)
print(f"Margin check (UnitPrice >= StandardCost) : success={margin['success']}, "
      f"violations={margin['result'].get('unexpected_count', 0)}")

# 2. Composite key uniqueness
compkey = ge.from_pandas(df).expect_compound_columns_to_be_unique(
    column_list=['Customer', 'OrderDate', 'product', 'OrderQuantity'],
)
print(f"Composite key unique                      : success={compkey['success']}, "
      f"duplicates={compkey['result'].get('unexpected_count', 0)}")

# 3. Monotonic OrderDate after sorting
#    GE's increasing-check needs a numeric column for datetime64 types,
#    so we cast to int64 nanoseconds-since-epoch first.
df_sorted = df.sort_values('OrderDate').reset_index(drop=True)
df_sorted['OrderDateInt'] = df_sorted['OrderDate'].astype('int64')
mono = ge.from_pandas(df_sorted).expect_column_values_to_be_increasing(
    column='OrderDateInt', strictly=False,
)
print(f"OrderDate monotonic after sort            : success={mono['success']}")

# 4. Inject a back-dated row and watch it fail
df_timewarp = df_sorted.copy()
df_timewarp.loc[df_timewarp.index[-1], 'OrderDateInt'] = pd.Timestamp('2010-01-01').value
warp = ge.from_pandas(df_timewarp).expect_column_values_to_be_increasing(
    column='OrderDateInt', strictly=False,
)
print(f"After back-dating the last row            : success={warp['success']}, "
      f"violations={warp['result'].get('unexpected_count', 0)}")

## Step 16 — String/format rules (regex, length, datetime parsing)

Most data warehouses are full of strings that *should* match a known shape: ZIP codes, state codes, phone numbers, email addresses, ISO dates, SKU identifiers. GE has dedicated expectations for all of them.

We'll show three on real columns of the dataset:
- **`expect_column_values_to_match_regex`** — US ZIP must be exactly 5 digits. *(This will surface a real-world bug in our data — leading zeros lost during ingestion!)*
- **`expect_column_value_lengths_to_equal`** — `StateCD` must be exactly 2 characters.
- **`expect_column_values_to_match_strftime_format`** — when dates arrive as strings (CSV/JSON), enforce the format.

In [ ]:
# 1. ZIP regex — values must be 5 digits. ZIPs are stored as ints in this file,
#    GE coerces them to strings before applying the regex.
ge_clean = ge.from_pandas(df)
zip_ok = ge_clean.expect_column_values_to_match_regex('zip', r'^\d{5}$')
print(f"ZIP matches ^\\d{{5}}$  : success={zip_ok['success']}, "
      f"violations={zip_ok['result'].get('unexpected_count', 0)}")
if not zip_ok['success']:
    samples = zip_ok['result'].get('partial_unexpected_list', [])[:5]
    print(f"  Sample bad ZIPs       : {samples}  <- 4-digit values: leading zero lost during ingest")
    print('  (Real Massachusetts ZIPs like 02134 became 2134 when the column was cast to int.)')

# 2. State code length
state_ok = ge_clean.expect_column_value_lengths_to_equal('StateCD', value=2)
print(f"\nStateCD length == 2   : success={state_ok['success']}, "
      f"violations={state_ok['result'].get('unexpected_count', 0)}")

# 3. Date strings: simulate CSV ingest where OrderDate arrives as text
df_csv = df.copy()
df_csv['OrderDateStr'] = df_csv['OrderDate'].dt.strftime('%Y-%m-%d')
fmt_ok = ge.from_pandas(df_csv).expect_column_values_to_match_strftime_format(
    'OrderDateStr', strftime_format='%Y-%m-%d',
)
print(f"OrderDateStr is %Y-%m-%d : success={fmt_ok['success']}")

# 4. Inject malformed strings and watch it fail
df_csv.loc[df_csv.index[:4], 'OrderDateStr'] = ['2024/01/01', '01-15-2024', 'yesterday', None]
fmt_fail = ge.from_pandas(df_csv).expect_column_values_to_match_strftime_format(
    'OrderDateStr', strftime_format='%Y-%m-%d',
)
print(f"After 4 malformed dates : success={fmt_fail['success']}, "
      f"violations={fmt_fail['result'].get('unexpected_count', 0)}, "
      f"sample={fmt_fail['result'].get('partial_unexpected_list', [])[:4]}")

## Step 17 — Custom expectation (your own business rule)

When the built-in 50+ expectations don't cover a domain rule, you write your own and it gets first-class treatment: it shows up in Data Docs, returns `unexpected_count` and `partial_unexpected_list`, supports `mostly=`, etc.

We'll codify a **revenue audit** rule: in a clean dataset, `Sales` should equal `UnitPrice × OrderQuantity` (within rounding). This is exactly the kind of check finance/audit teams need but no built-in expectation provides.

We subclass `PandasDataset` and decorate a method with `@MetaPandasDataset.multicolumn_map_expectation` — GE handles the rest (parameter validation, result formatting, sample collection).

In [ ]:
from great_expectations.dataset import PandasDataset, MetaPandasDataset

class SalesDataset(PandasDataset):
    """PandasDataset extended with a domain-specific revenue-integrity rule."""
    _data_asset_type = 'SalesDataset'

    @MetaPandasDataset.multicolumn_map_expectation
    def expect_sales_to_equal_unitprice_times_quantity(self, column_list, tolerance=0.5):
        # column_list arrives as a DataFrame slice with the requested columns, in order
        unit_price = column_list.iloc[:, 0]
        quantity   = column_list.iloc[:, 1]
        sales      = column_list.iloc[:, 2]
        # Must return a boolean Series: True = row passes
        return (sales - unit_price * quantity).abs() < tolerance

print('Defined SalesDataset with custom expectation:')
print('  - expect_sales_to_equal_unitprice_times_quantity')

In [ ]:
# Restrict to non-discounted rows so the rule (Sales == UnitPrice*Qty) holds exactly.
# Discounted rows have Sales < UnitPrice*Qty by design.
no_discount = df[df['Discount'] == 0].reset_index(drop=True)
sales_ds = ge.from_pandas(no_discount, dataset_class=SalesDataset)

clean = sales_ds.expect_sales_to_equal_unitprice_times_quantity(
    column_list=['UnitPrice', 'OrderQuantity', 'Sales'],
)
print(f"Clean data (Discount==0) : success={clean['success']}, "
      f"violations={clean['result'].get('unexpected_count', 0)}/{len(no_discount):,}")

# Inject 3 fraudulent / corrupted rows where Sales has been tampered with
df_audit = no_discount.copy()
df_audit.loc[df_audit.index[0], 'Sales'] = df_audit.loc[df_audit.index[0], 'Sales'] + 100
df_audit.loc[df_audit.index[1], 'Sales'] = 0
df_audit.loc[df_audit.index[2], 'Sales'] = -50
audit_ds = ge.from_pandas(df_audit, dataset_class=SalesDataset)

tampered = audit_ds.expect_sales_to_equal_unitprice_times_quantity(
    column_list=['UnitPrice', 'OrderQuantity', 'Sales'],
    result_format='COMPLETE',
)
print(f"\nTampered data            : success={tampered['success']}, "
      f"violations={tampered['result'].get('unexpected_count', 0)}")
print('Bad rows (UnitPrice, OrderQuantity, Sales):')
for row in tampered['result'].get('partial_unexpected_list', [])[:5]:
    print(f"  {row}")
print(f"Bad row indices: {tampered['result'].get('partial_unexpected_index_list', [])[:5]}")

# The custom expectation can be persisted into a suite just like any built-in one
audit_ds.save_expectation_suite('sales.audit.expectations.json',
                                discard_failed_expectations=False)
print('\nSaved -> sales.audit.expectations.json   (custom expectation persisted)')

## Recap

**Part 1 — Core workflow**

| Step | What we did | Key API |
|------|-------------|---------|
| 1 | Installed GE + deps | `pip install great_expectations==0.15.41 pandas openpyxl` |
| 2 | Loaded data | `pd.read_excel('data.xlsx')` |
| 3 | Wrapped DataFrame | `ge.from_pandas(df)` |
| 4 | Table check | `expect_table_row_count_to_be_between` |
| 5 | Column checks | `expect_column_to_exist`, `..._values_to_not_be_null`, `..._in_type_list` |
| 6 | Categorical check | `expect_column_values_to_be_in_set` |
| 7 | Numeric ranges | `..._values_to_be_between`, `..._mean_to_be_between` |
| 8 | Saved suite | `save_expectation_suite('sales.data.expectations.json')` |
| 9 | Re-validated | `df.validate(expectation_suite=...)` |
| 10 | Pipeline gate | branch on `results['success']` |
| 11 | Visual report (Data Docs) | `BaseDataContext` + `Checkpoint` + `build_data_docs()` |
| 12 | Stress test with injected failures | `df.copy()` + `validate(expectation_suite=...)` |

**Part 2 — Advanced features**

| Step | What we did | Key API |
|------|-------------|---------|
| 13 | Auto-generated suite from data | `UserConfigurableProfiler(...).build_suite()` |
| 14 | Distribution drift detection | `expect_column_kl_divergence_to_be_less_than` + `build_continuous_partition_object` |
| 15 | Cross-column integrity | `expect_column_pair_values_A_to_be_greater_than_B`, `expect_compound_columns_to_be_unique`, `expect_column_values_to_be_increasing` |
| 16 | String/format rules | `expect_column_values_to_match_regex`, `expect_column_value_lengths_to_equal`, `expect_column_values_to_match_strftime_format` |
| 17 | Custom domain expectation | subclass `PandasDataset` + `@MetaPandasDataset.multicolumn_map_expectation` |

**Next steps to explore:**
- **GX Cloud** — hosted multi-user web UI (separate notebook in this folder)
- The modern **Fluent Datasources / Checkpoints** API in GE ≥ 1.x
- Connecting to **SQL databases**, **Spark**, **S3**, **BigQuery** as data sources
- **Checkpoint actions**: Slack/Teams/Email notifications on failure
- **Conditional expectations** with `row_condition=` for per-segment rules